# Evaluation frameworks: RAGAS: the real-library code of the lessons

Booster LEAP, Advanced RAG Engineering. The cells below are the library versions of the examples in the lessons (the lessons first build the same mechanism in plain Python in your browser). Each cell was run once and its output is saved below it.

<h1 style="color:#b45309">&#9888; First: File &rarr; Save a copy in Drive</h1>

<span style="font-size:20px">This notebook is read-only. Make your own copy now, then work in the copy (no GitHub login needed).</span>

**How to use it:** run the cells from top to bottom (Runtime, Run all, or Shift+Enter on each cell). Change one thing at a time and run again.

**What needs what:** your own AWS key with Bedrock access; no GPU. The saved outputs are from a real run on 27 November 2026 and your scores will differ. The cells call real models on Amazon Bedrock, so you need your own AWS account with access to the models. The first lines of a cell ask for your own access key with getpass; the key is not stored in the notebook. The calls are billed to your account. The install line pins the versions that were used to produce the saved outputs; Colab may resolve the dependencies of those packages differently.

In [ ]:
%pip -q install ragas==0.4.4 litellm==1.111.0 instructor==1.18.0 boto3==1.44.12

## Faithfulness and answer grounding

Saved output of a real run (27 November 2026, ragas 0.4.4, judge Amazon Nova Lite). Not run on this page; you need your own key.

In [ ]:
import asyncio, os
from getpass import getpass

os.environ["RAGAS_DO_NOT_TRACK"] = "true"        # no usage reports from the library
if "AWS_PROFILE" not in os.environ and "AWS_ACCESS_KEY_ID" not in os.environ:
    key = getpass("Your AWS access key id (press Enter if you have no AWS account): ").strip()
    if not key:
        raise RuntimeError("These cells call Amazon Bedrock, so they need your own AWS account. "
                           "Without one, read the saved output shown on the lesson page.")
    os.environ["AWS_ACCESS_KEY_ID"] = key
    os.environ["AWS_SECRET_ACCESS_KEY"] = getpass("Your AWS secret access key: ").strip()
    token = getpass("Session token (temporary keys only, otherwise press Enter): ").strip()
    if token:
        os.environ["AWS_SESSION_TOKEN"] = token
os.environ["AWS_REGION_NAME"] = "ap-south-1"

import instructor, litellm
from ragas.llms import llm_factory

llm = llm_factory("bedrock/apac.amazon.nova-lite-v1:0", provider="litellm",
                  client=litellm.acompletion, adapter="instructor", temperature=0.0,
                  mode=instructor.Mode.MD_JSON)   # accepts JSON wrapped in a code fence

from ragas.metrics.collections import Faithfulness

uptime = ("The Business plan commits to 99.9 percent monthly uptime, with a credit "
          "of 10 percent of the month's fee if it falls below.")
trash = "Deleted tickets stay in the trash for 30 days and are then removed for good."
records = [
    ("What happens if the Business plan misses its uptime commitment?",
     "The Business plan commits to 99.9 percent monthly uptime. If uptime falls below "
     "that, you get a credit of 10 percent of the month's fee. The credit is applied "
     "automatically within 7 days.", [uptime]),
    ("How long do deleted tickets stay in the trash?",
     "Deleted tickets stay in the trash for 60 days and are then removed for good.", [trash]),
]

async def main():
    metric = Faithfulness(llm=llm)
    for question, answer, contexts in records:
        result = await metric.ascore(user_input=question, response=answer,
                                     retrieved_contexts=contexts)
        print(f"{result.value:.3f}  {question}")

await main()

0.667  What happens if the Business plan misses its uptime commitment?
0.000  How long do deleted tickets stay in the trash?


## Context recall and context precision

Saved output of a real run (27 November 2026, ragas 0.4.4, judge Amazon Nova Lite). Not run on this page; you need your own key.

In [ ]:
import asyncio, os
from getpass import getpass

os.environ["RAGAS_DO_NOT_TRACK"] = "true"        # no usage reports from the library
if "AWS_PROFILE" not in os.environ and "AWS_ACCESS_KEY_ID" not in os.environ:
    key = getpass("Your AWS access key id (press Enter if you have no AWS account): ").strip()
    if not key:
        raise RuntimeError("These cells call Amazon Bedrock, so they need your own AWS account. "
                           "Without one, read the saved output shown on the lesson page.")
    os.environ["AWS_ACCESS_KEY_ID"] = key
    os.environ["AWS_SECRET_ACCESS_KEY"] = getpass("Your AWS secret access key: ").strip()
    token = getpass("Session token (temporary keys only, otherwise press Enter): ").strip()
    if token:
        os.environ["AWS_SESSION_TOKEN"] = token
os.environ["AWS_REGION_NAME"] = "ap-south-1"

import instructor, litellm
from ragas.llms import llm_factory

llm = llm_factory("bedrock/apac.amazon.nova-lite-v1:0", provider="litellm",
                  client=litellm.acompletion, adapter="instructor", temperature=0.0,
                  mode=instructor.Mode.MD_JSON)   # accepts JSON wrapped in a code fence

from ragas.metrics.collections import ContextPrecisionWithReference, ContextRecall

parental = ("The primary caregiver receives 26 weeks of paid parental leave and the "
            "second caregiver receives 6 weeks.")
annual = "Full-time employees receive 24 days of paid annual leave per calendar year."
carry = "Up to 5 unused annual leave days may be carried over and must be used by 31 March."
question = "How much parental leave does each caregiver get, and how early must I request it?"
reference = ("The primary caregiver gets 26 weeks. The second caregiver gets 6 weeks. Leave is "
             "requested in the HR portal at least 5 working days before the first day.")

async def main():
    recall = ContextRecall(llm=llm)
    precision = ContextPrecisionWithReference(llm=llm)
    for name, contexts in [("useful passage first", [parental, annual, carry]),
                           ("useful passage last", [annual, carry, parental])]:
        r = await recall.ascore(user_input=question, retrieved_contexts=contexts,
                                reference=reference)
        p = await precision.ascore(user_input=question, reference=reference,
                                   retrieved_contexts=contexts)
        print(f"{name:22s} recall {r.value:.3f}  precision {p.value:.3f}")

await main()

useful passage first   recall 0.667  precision 0.000
useful passage last    recall 0.667  precision 0.333


## Answer relevancy

Saved output of a real run (27 November 2026, ragas 0.4.4, Amazon Nova Lite and Titan Text Embeddings V2). Not run on this page; you need your own key.

In [ ]:
import asyncio, os
from getpass import getpass

os.environ["RAGAS_DO_NOT_TRACK"] = "true"        # no usage reports from the library
if "AWS_PROFILE" not in os.environ and "AWS_ACCESS_KEY_ID" not in os.environ:
    key = getpass("Your AWS access key id (press Enter if you have no AWS account): ").strip()
    if not key:
        raise RuntimeError("These cells call Amazon Bedrock, so they need your own AWS account. "
                           "Without one, read the saved output shown on the lesson page.")
    os.environ["AWS_ACCESS_KEY_ID"] = key
    os.environ["AWS_SECRET_ACCESS_KEY"] = getpass("Your AWS secret access key: ").strip()
    token = getpass("Session token (temporary keys only, otherwise press Enter): ").strip()
    if token:
        os.environ["AWS_SESSION_TOKEN"] = token
os.environ["AWS_REGION_NAME"] = "ap-south-1"

import instructor, litellm
from ragas.llms import llm_factory

llm = llm_factory("bedrock/apac.amazon.nova-lite-v1:0", provider="litellm",
                  client=litellm.acompletion, adapter="instructor", temperature=0.0,
                  mode=instructor.Mode.MD_JSON)   # accepts JSON wrapped in a code fence

from ragas.embeddings import LiteLLMEmbeddings
from ragas.metrics.collections import AnswerRelevancy

embeddings = LiteLLMEmbeddings(model="bedrock/amazon.titan-embed-text-v2:0")
records = [
    ("direct", "How many paid sick days can I take?",
     "You can take up to 10 paid sick days per calendar year."),
    ("off the question", "Can I book business class for a 4 hour flight?",
     "Claims above 5000 rupees need a receipt and manager approval."),
    ("evasive", "What is the dinner limit for a Tier 2 city?",
     "I am not sure. It may depend on your city, so please check the expense policy."),
]

async def main():
    metric = AnswerRelevancy(llm=llm, embeddings=embeddings)   # 3 questions per answer
    for name, question, answer in records:
        result = await metric.ascore(user_input=question, response=answer)
        print(f"{result.value:.3f}  {name}")

await main()

0.828  direct
0.080  off the question
0.000  evasive


## An LLM as the judge

A custom pass or fail judge. The output is saved from one real run, not run on this page; running it needs your own AWS key.

In [ ]:
import asyncio, os
from getpass import getpass

os.environ["RAGAS_DO_NOT_TRACK"] = "true"        # no usage reports from the library
if "AWS_PROFILE" not in os.environ and "AWS_ACCESS_KEY_ID" not in os.environ:
    key = getpass("Your AWS access key id (press Enter if you have no AWS account): ").strip()
    if not key:
        raise RuntimeError("These cells call Amazon Bedrock, so they need your own AWS account. "
                           "Without one, read the saved output shown on the lesson page.")
    os.environ["AWS_ACCESS_KEY_ID"] = key
    os.environ["AWS_SECRET_ACCESS_KEY"] = getpass("Your AWS secret access key: ").strip()
    token = getpass("Session token (temporary keys only, otherwise press Enter): ").strip()
    if token:
        os.environ["AWS_SESSION_TOKEN"] = token
os.environ["AWS_REGION_NAME"] = "ap-south-1"

import instructor, litellm
from ragas.llms import llm_factory

llm = llm_factory("bedrock/apac.amazon.nova-lite-v1:0", provider="litellm",
                  client=litellm.acompletion, adapter="instructor", temperature=0.0,
                  mode=instructor.Mode.MD_JSON)   # accepts JSON wrapped in a code fence

from ragas.metrics import DiscreteMetric

judge = DiscreteMetric(
    name="grounded",
    allowed_values=["pass", "fail"],
    prompt=("You check a support answer against a passage. Return fail if any fact or number "
            "in the answer is missing from the passage or differs from it. Otherwise return pass. "
            "Give a short reason.\nPassage: {context}\nAnswer: {answer}"),
)
cases = [
    ("added claim", "The Business plan commits to 99.9 percent monthly uptime, with a credit of "
     "10 percent of the month's fee if it falls below.",
     "You get a 10 percent credit, applied automatically within 7 days."),
    ("wrong number", "Deleted tickets stay in the trash for 30 days and are then removed for good.",
     "Deleted tickets stay in the trash for 60 days."),
    ("sound", "Error ERR-4021 at sign-in: the authenticator code is out of date. Check that the "
     "phone clock is set to automatic time.",
     "ERR-4021 means the authenticator code is out of date; set the phone clock to automatic."),
]

async def main():
    for name, context, answer in cases:
        result = await judge.ascore(llm=llm, context=context, answer=answer)
        print(f"{name}: {result.value}\n   {result.reason}")

await main()

added claim: fail
   The answer correctly states that a 10 percent credit is applied, but it does not mention the condition of the 99.9 percent monthly uptime or the timeframe of 7 days, which are present in the passage.
wrong number: fail
   The number of days deleted tickets stay in the trash differs between the passage and the answer.
sound: pass
   The answer correctly reflects the information in the passage. The error code ERR-4021 and the solution to set the phone clock to automatic time are both mentioned in the passage.


## Notebook: an end-to-end evaluation

Four metrics on two records. The output is saved from one real run, not run on this page; running it needs your own AWS key.

In [ ]:
import asyncio, os
from getpass import getpass

os.environ["RAGAS_DO_NOT_TRACK"] = "true"        # no usage reports from the library
if "AWS_PROFILE" not in os.environ and "AWS_ACCESS_KEY_ID" not in os.environ:
    key = getpass("Your AWS access key id (press Enter if you have no AWS account): ").strip()
    if not key:
        raise RuntimeError("These cells call Amazon Bedrock, so they need your own AWS account. "
                           "Without one, read the saved output shown on the lesson page.")
    os.environ["AWS_ACCESS_KEY_ID"] = key
    os.environ["AWS_SECRET_ACCESS_KEY"] = getpass("Your AWS secret access key: ").strip()
    token = getpass("Session token (temporary keys only, otherwise press Enter): ").strip()
    if token:
        os.environ["AWS_SESSION_TOKEN"] = token
os.environ["AWS_REGION_NAME"] = "ap-south-1"

import instructor, litellm
from ragas.llms import llm_factory

llm = llm_factory("bedrock/apac.amazon.nova-lite-v1:0", provider="litellm",
                  client=litellm.acompletion, adapter="instructor", temperature=0.0,
                  mode=instructor.Mode.MD_JSON)   # accepts JSON wrapped in a code fence

from ragas.embeddings import LiteLLMEmbeddings
from ragas.metrics.collections import (AnswerRelevancy, ContextPrecisionWithReference,
                                       ContextRecall, Faithfulness)

embeddings = LiteLLMEmbeddings(model="bedrock/amazon.titan-embed-text-v2:0")

def row(question, contexts, answer, reference):
    return dict(question=question, contexts=contexts, answer=answer, reference=reference)

rows = [
    row("Where is customer data stored?",
        ["Never copy customer data into chat tools, personal notes or public AI services.",
         "Customer data is stored in the Mumbai region by default. Business plan customers "
         "may choose Frankfurt."],
        "Customer data is stored in the Mumbai region by default.",
        "In the Mumbai region by default; Business plan customers may choose Frankfurt."),
    row("How long do deleted tickets stay in the trash?",
        ["Deleted tickets stay in the trash for 30 days and are then removed for good."],
        "Deleted tickets stay in the trash for 60 days and are then removed for good.",
        "Deleted tickets stay in the trash for 30 days and are then removed for good."),
]

async def main():
    f, ar = Faithfulness(llm=llm), AnswerRelevancy(llm=llm, embeddings=embeddings)
    cr, cp = ContextRecall(llm=llm), ContextPrecisionWithReference(llm=llm)
    print("faithful  relevancy  recall  precision  question")
    for x in rows:
        q, c, a, ref = x["question"], x["contexts"], x["answer"], x["reference"]
        s = [await f.ascore(user_input=q, response=a, retrieved_contexts=c),
             await ar.ascore(user_input=q, response=a),
             await cr.ascore(user_input=q, retrieved_contexts=c, reference=ref),
             await cp.ascore(user_input=q, reference=ref, retrieved_contexts=c)]
        print("{:8.3f}  {:9.3f}  {:6.3f}  {:9.3f}  ".format(*[r.value for r in s]) + q)

await main()

faithful  relevancy  recall  precision  question
   1.000      0.913   1.000      0.500  Where is customer data stored?
   0.000      0.964   1.000      1.000  How long do deleted tickets stay in the trash?
